# Alignment review (optional, thin viewer)

**This notebook does no alignment.** It only flips through the batch-generated per-slide
`report.pdf` / `report.png` panels (from `run_report.py` / `slurm/report_array.sbatch`) and
collects a human score into `alignment_validation.csv`. All computation lives in
`hest_valis/report.py`; this is just a viewer + a scoring form (cf. Tyler Lam's
`check_alignment.ipynb`).

**Prerequisite:** run the batch report first so each `<output>/<slide>/report.png` exists:
```bash
sbatch --array=0-$(( $(tail -n +2 samples.csv | wc -l) - 1 )) slurm/report_array.sbatch
# or one slide:  python run_report.py --samples samples.csv --config config.json --sample <id>
```

**Score columns:** `slide, chosen, aligned (0=bad / 1=partial / 2=good), has_fold (0/1), note`.

In [ ]:
import os, glob, json, csv
from IPython.display import Image, display, clear_output

PIPE = os.path.dirname(os.path.abspath("."))  # repo root (run this notebook from notebooks/)
PIPE = PIPE if os.path.exists(os.path.join(PIPE, "hest_valis")) else \
       "/common/ruppinelab/Eldad/st_ref_v2/hest_valis_pipeline"
CONFIG = os.path.join(PIPE, "config.json")
OUTPUT_DIR = json.load(open(CONFIG))["output_dir"] if os.path.exists(CONFIG) else os.path.join(PIPE, "output")
if not os.path.isabs(OUTPUT_DIR):
    OUTPUT_DIR = os.path.normpath(os.path.join(PIPE, OUTPUT_DIR))
CSV_PATH = os.path.join(OUTPUT_DIR, "alignment_validation.csv")

# slides with a rendered report (report_array.sbatch / run_report.py produced report.png/pdf)
slides = []
for sd in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*"))):
    if not os.path.isdir(sd):
        continue
    png, pdf, qc = (os.path.join(sd, f) for f in ("report.png", "report.pdf", "qc.json"))
    if not (os.path.exists(png) or os.path.exists(pdf)):
        continue
    chosen = None
    if os.path.exists(qc):
        try:
            chosen = json.load(open(qc)).get("decision", {}).get("chosen")
        except Exception:
            pass
    slides.append({"slide": os.path.basename(sd), "dir": sd,
                   "png": png if os.path.exists(png) else None,
                   "pdf": pdf if os.path.exists(pdf) else None, "chosen": chosen})
print(f"{len(slides)} slides with a rendered report under {OUTPUT_DIR}")

# resume prior scores if the CSV already exists
scores = {}
if os.path.exists(CSV_PATH):
    for r in csv.DictReader(open(CSV_PATH)):
        scores[r["slide"]] = r
    print(f"loaded {len(scores)} existing scores from {CSV_PATH}")


def save_scores():
    """Write alignment_validation.csv (slide, chosen, aligned, has_fold, note)."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    cols = ["slide", "chosen", "aligned", "has_fold", "note"]
    with open(CSV_PATH, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for s in slides:
            r = scores.get(s["slide"], {})
            w.writerow({"slide": s["slide"], "chosen": s["chosen"],
                        "aligned": r.get("aligned", ""), "has_fold": r.get("has_fold", ""),
                        "note": r.get("note", "")})
    print("wrote", CSV_PATH)

## Interactive reviewer

Use **prev / next** to flip slides, set **aligned / has_fold / note**, then **save CSV**
(saves incrementally; you can stop and resume any time). If `ipywidgets` is not installed,
skip to the *Manual fallback* cell below.

In [ ]:
try:
    import ipywidgets as W
    HAVE_W = True
except Exception:
    HAVE_W = False
    print("ipywidgets unavailable -> use the Manual fallback cell below")

if HAVE_W and slides:
    idx = {"i": 0}
    out = W.Output()
    aligned = W.ToggleButtons(options=[("bad (0)", 0), ("partial (1)", 1), ("good (2)", 2)],
                              description="aligned")
    fold = W.Checkbox(value=False, description="has_fold")
    note = W.Text(description="note", layout=W.Layout(width="60%"))
    prev_b = W.Button(description="< prev")
    next_b = W.Button(description="next >")
    save_b = W.Button(description="save CSV", button_style="success")

    def _load_form(s):
        r = scores.get(s["slide"], {})
        aligned.value = int(r["aligned"]) if str(r.get("aligned", "")).strip().isdigit() else 1
        fold.value = str(r.get("has_fold", "")).strip().lower() in ("1", "true", "yes")
        note.value = r.get("note", "") or ""

    def _render():
        s = slides[idx["i"]]
        with out:
            clear_output(wait=True)
            print(f"[{idx['i']+1}/{len(slides)}] {s['slide']}  (chosen={s['chosen']})")
            if s["png"]:
                display(Image(filename=s["png"], width=1100))
            else:
                print("no report.png -- open the PDF:", s["pdf"])
        _load_form(s)

    def _stash():
        s = slides[idx["i"]]
        scores[s["slide"]] = {"slide": s["slide"], "chosen": s["chosen"],
                              "aligned": aligned.value, "has_fold": int(fold.value),
                              "note": note.value}

    def _go(delta):
        _stash()
        idx["i"] = max(0, min(len(slides) - 1, idx["i"] + delta))
        _render()

    prev_b.on_click(lambda b: _go(-1))
    next_b.on_click(lambda b: _go(1))
    save_b.on_click(lambda b: (_stash(), save_scores()))
    display(W.VBox([W.HBox([prev_b, next_b, save_b]), aligned, fold, note, out]))
    _render()

## Manual fallback (no ipywidgets)

Open each `<output>/<slide>/report.pdf`, then fill the dict and run `save_scores()`.

In [ ]:
# seed empty rows, then edit and save
for s in slides:
    scores.setdefault(s["slide"], {"slide": s["slide"], "chosen": s["chosen"],
                                   "aligned": "", "has_fold": "", "note": ""})

# example edits:
# scores["P10"].update(aligned=2, has_fold=0, note="clean")
# scores["P28"].update(aligned=0, has_fold=1, note="90-deg rotation not recovered")

# save_scores()

In [ ]:
# final: write + preview
save_scores()
import pandas as pd
pd.read_csv(CSV_PATH)